# Feedwater System Load Correlation

From the [Sisyphean Gridworks ML Playground](https://sgridworks.com/ml-playground/guides/18-feedwater-load-correlation.html)

## Setup

Clone the repository and install dependencies. Run this cell first.

In [ ]:
import os, subprocess

# Colab: clone the repo and cd into it
# Local: detect if we are already inside the repo
if not os.path.exists('sisyphean-power-and-light'):
    if os.path.exists('../sisyphean-power-and-light'):
        os.chdir('..')  # running from notebooks/ subfolder
    else:
        subprocess.run(['git', 'clone', 'https://github.com/SGridworks/Dynamic-Network-Model.git'], capture_output=True)
        os.chdir('Dynamic-Network-Model')

print(f'Working directory: {os.getcwd()}')
# !pip install -q pandas numpy matplotlib seaborn scikit-learn pyarrow


## Step 0: Verify Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split

# Load BFP hourly data
DATA_PATH = "sisyphean-power-and-light/generation/timeseries/bfp_train_hourly.parquet"
df = pd.read_parquet(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.set_index("timestamp").sort_index()

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
assert len(df) == 8784, f"Expected 8,784 rows, got {len(df)}"
print("\nSetup verified.")

## Step 1: Load BFP and System Data

The feedwater system operates in lockstep with unit load. When the gas turbines produce more power, the HRSG needs more steam, which requires more feedwater flow. The BFP motor power, discharge pressure, and flow should all track with `U1_UNIT_MW_GROSS`.

In [ ]:
# Key system columns
system_cols = ["U1_UNIT_MW_GROSS", "U1_AMBIENT_TEMP", "U1_FW_HDR_FLOW",
               "U1_DA_PRESS", "U1_DA_LEVEL"]

# BFP-A operating columns
bfpa_cols = ["U1_BFPA_FW_FLOW", "U1_BFPA_MTR_POWER", "U1_BFPA_DISCH_PRESS",
             "U1_BFPA_MTR_CURRENT", "U1_BFPA_SPEED", "U1_BFPA_RUN_STATUS"]

# BFP-B operating columns
bfpb_cols = ["U1_BFPB_FW_FLOW", "U1_BFPB_MTR_POWER", "U1_BFPB_DISCH_PRESS",
             "U1_BFPB_MTR_CURRENT", "U1_BFPB_SPEED", "U1_BFPB_RUN_STATUS"]

# Create a unified "active pump" view
# At any given time, only one pump is running (except brief overlaps during swaps)
df["active_pump"] = "none"
df.loc[df["U1_BFPA_RUN_STATUS"] > 0, "active_pump"] = "A"
df.loc[df["U1_BFPB_RUN_STATUS"] > 0, "active_pump"] = "B"

# Create unified columns for the running pump
df["BFP_FW_FLOW"] = np.where(df["active_pump"] == "A",
                              df["U1_BFPA_FW_FLOW"], df["U1_BFPB_FW_FLOW"])
df["BFP_MTR_POWER"] = np.where(df["active_pump"] == "A",
                                df["U1_BFPA_MTR_POWER"], df["U1_BFPB_MTR_POWER"])
df["BFP_DISCH_PRESS"] = np.where(df["active_pump"] == "A",
                                  df["U1_BFPA_DISCH_PRESS"], df["U1_BFPB_DISCH_PRESS"])

print("Active pump distribution:")
print(df["active_pump"].value_counts())
print(f"\nSystem MW range: {df['U1_UNIT_MW_GROSS'].min():.0f} - {df['U1_UNIT_MW_GROSS'].max():.0f} MW")

## Step 2: Explore Correlations

Scatter plots reveal the relationship between unit load (MW) and the three key BFP parameters: feedwater flow, motor power, and discharge pressure. In a healthy pump, these relationships should be tight and predictable.

In [ ]:
# Filter to running pump only
running = df[df["active_pump"] != "none"].copy()
print(f"Running hours: {len(running):,}")

fig, axes = plt.subplots(1, 3, figsize=(10, 5))

targets = [
    ("BFP_FW_FLOW", "FW Flow (t/hr)"),
    ("BFP_MTR_POWER", "Motor Power (kW)"),
    ("BFP_DISCH_PRESS", "Discharge Press (bar)"),
]

for ax, (col, label) in zip(axes, targets):
    ax.scatter(running["U1_UNIT_MW_GROSS"], running[col],
               c="#5FCCDB", s=2, alpha=0.3)
    ax.set_xlabel("Unit MW Gross")
    ax.set_ylabel(label)
    ax.set_title(f"MW vs {label}")

plt.tight_layout()
plt.show()

# Correlation matrix
corr_cols = ["U1_UNIT_MW_GROSS", "BFP_FW_FLOW", "BFP_MTR_POWER",
             "BFP_DISCH_PRESS", "U1_AMBIENT_TEMP"]
print("\nCorrelation matrix:")
print(running[corr_cols].corr().round(3).to_string())

## Step 3: Filter to Running Pump Only

We already filtered to running hours. Now let's color-code by which pump is active and by time period to see if the MW-to-flow relationship changes between pumps or over time (which would indicate degradation).

In [ ]:
# Color by pump and time period
fig, ax = plt.subplots(figsize=(8, 5))

pump_a = running[running["active_pump"] == "A"]
pump_b = running[running["active_pump"] == "B"]

ax.scatter(pump_a["U1_UNIT_MW_GROSS"], pump_a["BFP_MTR_POWER"],
           c="#5FCCDB", s=3, alpha=0.3, label="BFP-A")
ax.scatter(pump_b["U1_UNIT_MW_GROSS"], pump_b["BFP_MTR_POWER"],
           c="#D69E2E", s=3, alpha=0.3, label="BFP-B")
ax.set_xlabel("Unit MW Gross")
ax.set_ylabel("BFP Motor Power (kW)")
ax.set_title("Motor Power vs Unit Load: A vs B")
ax.legend()
plt.tight_layout()
plt.show()

# Check if there are visible differences in the MW-power relationship
# between early (healthy) and late (fault) periods for BFP-A
fig, ax = plt.subplots(figsize=(8, 5))

early = pump_a[pump_a.index < "2024-04-01"]
mid   = pump_a[(pump_a.index >= "2024-04-01") & (pump_a.index < "2024-06-01")]
late  = pump_a[pump_a.index >= "2024-10-15"]

ax.scatter(early["U1_UNIT_MW_GROSS"], early["BFP_MTR_POWER"],
           c="#5FCCDB", s=3, alpha=0.3, label="Jan-Mar (healthy)")
ax.scatter(mid["U1_UNIT_MW_GROSS"], mid["BFP_MTR_POWER"],
           c="#D69E2E", s=3, alpha=0.3, label="Apr-May (seal fault)")
ax.scatter(late["U1_UNIT_MW_GROSS"], late["BFP_MTR_POWER"],
           c="#E53E3E", s=3, alpha=0.3, label="Oct-Dec (misalignment)")
ax.set_xlabel("Unit MW Gross")
ax.set_ylabel("BFP-A Motor Power (kW)")
ax.set_title("BFP-A: Motor Power vs Load by Condition Period")
ax.legend()
plt.tight_layout()
plt.show()

## Step 4: Simple Linear Regression -- Unit MW to Feedwater Flow

The most fundamental relationship in the feedwater system: higher unit load requires more feedwater. We train a simple linear regression on healthy data (Jan-Mar) and use it to predict expected flow at any load.

In [ ]:
# Train on healthy BFP-A data (Jan-Mar)
healthy = running[(running.index < "2024-04-01") & (running["active_pump"] == "A")].copy()
print(f"Healthy training samples: {len(healthy):,}")

X_train = healthy[["U1_UNIT_MW_GROSS"]]
y_train = healthy["BFP_FW_FLOW"]

lr_simple = LinearRegression()
lr_simple.fit(X_train, y_train)

print(f"\nSimple Linear Regression: FW_FLOW = {lr_simple.coef_[0]:.3f} * MW + {lr_simple.intercept_:.1f}")
print(f"R-squared (training): {lr_simple.score(X_train, y_train):.4f}")

# Plot the fit
fig, ax = plt.subplots(figsize=(8, 5))
mw_range = np.linspace(running["U1_UNIT_MW_GROSS"].min(),
                       running["U1_UNIT_MW_GROSS"].max(), 100)
ax.scatter(healthy["U1_UNIT_MW_GROSS"], healthy["BFP_FW_FLOW"],
           c="#5FCCDB", s=3, alpha=0.3, label="Healthy data")
ax.plot(mw_range, lr_simple.predict(mw_range.reshape(-1, 1)),
        color="#2D6A7A", linewidth=2, label="Linear fit")
ax.set_xlabel("Unit MW Gross")
ax.set_ylabel("Feedwater Flow (t/hr)")
ax.set_title("Simple Linear Regression: MW to FW Flow")
ax.legend()
plt.tight_layout()
plt.show()

## Step 5: Multiple Regression -- Adding Ambient Temperature

Ambient temperature affects gas turbine output and HRSG heat transfer. Including it as a second predictor should improve the model for motor power and discharge pressure predictions.

In [ ]:
# Multiple regression: MW + Ambient Temp -> [FW_FLOW, MTR_POWER, DISCH_PRESS]
feature_cols = ["U1_UNIT_MW_GROSS", "U1_AMBIENT_TEMP"]
target_cols = ["BFP_FW_FLOW", "BFP_MTR_POWER", "BFP_DISCH_PRESS"]

X_healthy = healthy[feature_cols]
models = {}
results = []

print("Multiple Regression Results (trained on Jan-Mar healthy data):")
print("=" * 65)

for target in target_cols:
    y = healthy[target]
    model = LinearRegression()
    model.fit(X_healthy, y)
    y_pred = model.predict(X_healthy)
    r2 = r2_score(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    models[target] = model
    results.append({"Target": target, "R2": r2, "MAE": mae})
    print(f"\n{target}:")
    print(f"  Coefficients: MW={model.coef_[0]:.4f}, Ambient={model.coef_[1]:.4f}")
    print(f"  Intercept: {model.intercept_:.2f}")
    print(f"  R-squared: {r2:.4f}")
    print(f"  MAE: {mae:.2f}")

results_df = pd.DataFrame(results)
print(f"\n{results_df.to_string(index=False)}")

## Step 6: Residual Analysis -- Spotting Degradation Over Time

The real power of a load-correlation model is not prediction accuracy -- it is what happens when the residuals (actual minus predicted) start growing. A healthy pump should have residuals centered near zero. If seal leakage causes efficiency loss, the pump needs more power for the same flow, and the motor power residual goes positive.

In [ ]:
# Predict expected values for ALL running data, then compute residuals
for target in target_cols:
    running[f"{target}_pred"] = models[target].predict(running[feature_cols])
    running[f"{target}_resid"] = running[target] - running[f"{target}_pred"]

# Plot residuals over time for motor power (the strongest degradation signal)
fig, axes = plt.subplots(3, 1, figsize=(10, 5), sharex=True)

for ax, target, label in zip(axes, target_cols,
    ["FW Flow Residual (t/hr)", "Motor Power Residual (kW)", "Disch Press Residual (bar)"]):
    resid_col = f"{target}_resid"
    ax.plot(running.index, running[resid_col], color="#5FCCDB", linewidth=0.4, alpha=0.6)
    # 24h rolling mean of residual
    roll_resid = running[resid_col].rolling(24, min_periods=1).mean()
    ax.plot(running.index, roll_resid, color="#2D6A7A", linewidth=1.5)
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    ax.set_ylabel(label, fontsize=8)

    # Shade known fault periods
    ax.axvspan(pd.Timestamp("2024-04-01"), pd.Timestamp("2024-06-01"),
               alpha=0.1, color="orange", label="Seal fault")
    ax.axvspan(pd.Timestamp("2024-10-15"), pd.Timestamp("2024-12-31"),
               alpha=0.1, color="red", label="Misalignment")

axes[0].set_title("Residuals Over Time (actual - predicted from healthy baseline)")
axes[0].legend(fontsize=7, loc="upper right")
axes[-1].set_xlabel("Date")

plt.tight_layout()
plt.show()

# Monthly residual statistics
running["month"] = running.index.month
monthly_resid = running.groupby("month")["BFP_MTR_POWER_resid"].agg(["mean", "std"])
print("Monthly Motor Power Residual Stats:")
print(monthly_resid.round(2).to_string())

## Step 7: Cross-Reference with Grid Demand

SP&L's BFP parameters are driven by unit load, which in turn responds to grid demand. Load the distribution-side substation load data and show the correlation chain: grid demand drives unit MW, which drives BFP operation.

In [ ]:
# Load distribution-side substation load data
sub_load = pd.read_parquet(
    "sisyphean-power-and-light/timeseries/substation_load_hourly.parquet"
)
sub_load["timestamp"] = pd.to_datetime(sub_load["timestamp"])

# Aggregate total grid demand per hour across all substations
grid_demand = sub_load.groupby("timestamp")["total_load_mw"].sum().reset_index()
grid_demand = grid_demand.set_index("timestamp").sort_index()
grid_demand.columns = ["grid_demand_mw"]

print(f"Grid demand records: {len(grid_demand):,}")
print(f"Date range: {grid_demand.index.min()} to {grid_demand.index.max()}")

# Join with BFP data on timestamp
merged = running.join(grid_demand, how="inner")
print(f"Matched records: {len(merged):,}")

# Correlation between grid demand and unit MW
corr = merged["grid_demand_mw"].corr(merged["U1_UNIT_MW_GROSS"])
print(f"\nCorrelation (grid demand vs unit MW): {corr:.3f}")

# Plot the chain: grid demand -> unit MW -> BFP flow
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].scatter(merged["grid_demand_mw"], merged["U1_UNIT_MW_GROSS"],
                c="#5FCCDB", s=2, alpha=0.3)
axes[0].set_xlabel("Total Grid Demand (MW)")
axes[0].set_ylabel("Unit MW Gross")
axes[0].set_title(f"Grid Demand vs Unit Load (r={corr:.3f})")

corr2 = merged["grid_demand_mw"].corr(merged["BFP_FW_FLOW"])
axes[1].scatter(merged["grid_demand_mw"], merged["BFP_FW_FLOW"],
                c="#2D6A7A", s=2, alpha=0.3)
axes[1].set_xlabel("Total Grid Demand (MW)")
axes[1].set_ylabel("BFP FW Flow (t/hr)")
axes[1].set_title(f"Grid Demand vs BFP Flow (r={corr2:.3f})")

plt.tight_layout()
plt.show()

In [ ]:
# Time series overlay: grid demand vs BFP flow for one week
week = merged["2024-03-01":"2024-03-07"]

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(week.index, week["grid_demand_mw"], color="#5FCCDB", linewidth=1.5,
         label="Grid Demand (MW)")
ax2.plot(week.index, week["BFP_FW_FLOW"], color="#2D6A7A", linewidth=1.5,
         label="BFP FW Flow (t/hr)")

ax1.set_xlabel("Date")
ax1.set_ylabel("Grid Demand (MW)", color="#5FCCDB")
ax2.set_ylabel("BFP FW Flow (t/hr)", color="#2D6A7A")
ax1.set_title("One Week: Grid Demand vs BFP Feedwater Flow (Mar 1-7)")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

plt.tight_layout()
plt.show()

## What You Built and Next Steps

In this guide you:

1. **Loaded** 8,784 hours of BFP and unit operating data
2. **Explored** the correlation between unit load (MW) and feedwater flow, motor power, and discharge pressure
3. **Filtered** to running-pump-only data and identified differences between pumps and fault periods
4. **Built** a simple linear regression (MW to FW flow) with high R-squared on healthy data
5. **Extended** to multiple regression (MW + ambient temp) predicting three BFP parameters simultaneously
6. **Analyzed residuals** over time, showing how seal faults and misalignment cause residuals to drift from zero
7. **Cross-referenced** with distribution grid demand data, connecting the generation-side BFP parameters to downstream load

The residual analysis approach is a foundational technique in condition monitoring. When the model is trained on healthy data, growing residuals become an early warning of equipment degradation -- often weeks before a threshold alarm fires.

**Next steps:**
- **Guide 19**: Multi-class fault diagnosis using Random Forest and SHAP analysis
- **Guide 20**: Physics-informed digital twin using OEM pump curves for performance tracking